In [1]:
import pandas as pd
import pickle
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
)
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import FeatureUnion
from geoai.utils_ds.preprocessing_ops import PreProcessingOperations
from sklearn.feature_selection import SelectKBest, f_classif
from geoai.utils_ml.model_ops import ModelOperations
from geoai.utils_geo.raster_ops import RasterOperations

preprocess_ops = PreProcessingOperations()
model_ops = ModelOperations()
raster_ops = RasterOperations()


# We will create a pipeline using the following steps:

1. **Load the data containing only the bands.**

1. **Compute indices**

1. **Bin and Categorize NDVI**

1. **Pipeline 1**

- `One hot encode NDVI_binary`

- `Ordinal encode NDVI_category`

- `Apply Log transformation to numerical features`

- `Do a Polynomial transformation`

5. **Pipeline 2**

- `Pipeline 1`

- `Scale to 0-1`

- `Apply LDA`

6. **Pipeline 3**

- `Pipeline 1`

- `Scale to 0-1`

- `Apply PCA`

7. **Pipeline 4**:

- `Combine Pipeline 1, Pipeline 2, Pipeline 3`

8. **Pipeline 5**:

- `Select Features`

- `Train a logisitic regression model`

#### Step 1

In [2]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train_encoded.csv")
y_test = pd.read_csv("csv_files/y_test_encoded.csv")

#### Step 2 and 3

In [3]:
X_train = raster_ops.indices_binary_category(X_train) 
X_test = raster_ops.indices_binary_category(X_test)
X_train.head()

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_categorized,NDVI_binary
0,350.0,542.0000,323.0,3277.0000,1975.3334,0.820556,-0.247826,0.002545,high_veg,veg
1,390.0,555.0000,380.0,3016.6667,1991.0000,0.776251,-0.204819,0.002227,high_veg,veg
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000,0.044101,0.098425,0.000127,low_veg,non_veg
3,363.2,546.5000,395.0,3244.5000,2052.0000,0.782937,-0.225149,0.002438,high_veg,veg
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000,0.031066,0.123170,0.000098,low_veg,non_veg


#### Step 4: Pipeline 1

In [4]:
#======================================================#
# for categorical features
one_hot_encoder_columns = ["NDVI_binary"]
ordinal_encoder_columns = ["NDVI_categorized"]
categories = [["low_veg", "medium_veg", "high_veg"]]

one_hot_transformer = OneHotEncoder(
    dtype=int, sparse_output=False
)  # Instantiate the one hot transformer

ordinal_transformer = OrdinalEncoder(
    categories=categories, dtype=int
)  # Instantiate the ordinal transformer
#======================================================#


#======================================================#
# for numerical features
numerical_columns = X_train.select_dtypes(include=["float64"]).columns.tolist()
log_trasformer = FunctionTransformer(func=np.log1p)  # Instantiate the log transformer
poly_transformer = PolynomialFeatures(
    degree=2, include_bias=False
)  # Instantiate the polynomial transformer
#======================================================#

#======================================================#
# make a pipeline for each type of transformer
categorical_transformer_1 = Pipeline(
    steps=[("one_hot_transformer", one_hot_transformer)]
)

categorical_transformer_2 = Pipeline(
    steps=[("ordinal_transformer", ordinal_transformer)]
)

numerical_transformer = Pipeline(
    steps=[("log", log_trasformer), ("poly", poly_transformer)]
)

#======================================================#
# Create a preprocessor that includes the numerical, one hot, and ordinal transformers
pipeline_1 = ColumnTransformer(
    transformers=[
        (
            "categorical_transformer_1",
            categorical_transformer_1,
            one_hot_encoder_columns,
        ),
        (
            "categorical_transformer_2",
            categorical_transformer_2,
            ordinal_encoder_columns,
        ),
        ("numerical_transformer", numerical_transformer, numerical_columns),
    ]
)
pipeline_1

ColumnTransformer(transformers=[('categorical_transformer_1',
                                 Pipeline(steps=[('one_hot_transformer',
                                                  OneHotEncoder(dtype=<class 'int'>,
                                                                sparse_output=False))]),
                                 ['NDVI_binary']),
                                ('categorical_transformer_2',
                                 Pipeline(steps=[('ordinal_transformer',
                                                  OrdinalEncoder(categories=[['low_veg',
                                                                              'medium_veg',
                                                                              'high_veg']],
                                                                 dtype=<class 'int'>))]),
                                 ['NDVI_categorized']),
                                ('numerical_transformer',
                                 Pipeline(steps=[('log',
                                                  FunctionTransformer(func=<ufunc 'log1p'>)),
                                                 ('poly',
                                                  PolynomialFeatures(include_bias=False))]),
                                 ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR', 'NDVI',
                                  'NDBI', 'REI'])])

#### Step 5: Pipeline 2

In [5]:
# Pipeline 2: Pipeline 1 + LDA
min_max_scaler = MinMaxScaler() # Instantiate MinMaxScaler
lda_transformer = LinearDiscriminantAnalysis(n_components=3) # Instantiate LDA

pipeline_2 = Pipeline(
    steps=[
        ("pipeline_1", pipeline_1),
        ("min_max_scaler", min_max_scaler),
        ("lda_transformer", lda_transformer),
    ]
)
pipeline_2

Pipeline(steps=[('pipeline_1',
                 ColumnTransformer(transformers=[('categorical_transformer_1',
                                                  Pipeline(steps=[('one_hot_transformer',
                                                                   OneHotEncoder(dtype=<class 'int'>,
                                                                                 sparse_output=False))]),
                                                  ['NDVI_binary']),
                                                 ('categorical_transformer_2',
                                                  Pipeline(steps=[('ordinal_transformer',
                                                                   OrdinalEncoder(categories=[['low_veg',
                                                                                               'medium_veg',
                                                                                               'high_veg']],
                                                                                  dtype=<class 'int'>))]),
                                                  ['NDVI_categorized']),
                                                 ('numerical_transformer',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('poly',
                                                                   PolynomialFeatures(include_bias=False))]),
                                                  ['BLUE', 'GREEN', 'RED',
                                                   'NIR', 'SWIR', 'NDVI',
                                                   'NDBI', 'REI'])])),
                ('min_max_scaler', MinMaxScaler()),
                ('lda_transformer',
                 LinearDiscriminantAnalysis(n_components=3))])

#### Step 6: Pipeline 3

In [6]:
# Pipeline 3: Pipeline 1 + PCA
pca_transformer = PCA(n_components=7)

pipeline_3 = Pipeline(
    steps=[
        ("pipeline_1", pipeline_1),
        ("min_max_scaler", min_max_scaler),
        ("pca_transformer", pca_transformer),
    ]
)
pipeline_3

Pipeline(steps=[('pipeline_1',
                 ColumnTransformer(transformers=[('categorical_transformer_1',
                                                  Pipeline(steps=[('one_hot_transformer',
                                                                   OneHotEncoder(dtype=<class 'int'>,
                                                                                 sparse_output=False))]),
                                                  ['NDVI_binary']),
                                                 ('categorical_transformer_2',
                                                  Pipeline(steps=[('ordinal_transformer',
                                                                   OrdinalEncoder(categories=[['low_veg',
                                                                                               'medium_veg',
                                                                                               'high_veg']],
                                                                                  dtype=<class 'int'>))]),
                                                  ['NDVI_categorized']),
                                                 ('numerical_transformer',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('poly',
                                                                   PolynomialFeatures(include_bias=False))]),
                                                  ['BLUE', 'GREEN', 'RED',
                                                   'NIR', 'SWIR', 'NDVI',
                                                   'NDBI', 'REI'])])),
                ('min_max_scaler', MinMaxScaler()),
                ('pca_transformer', PCA(n_components=7))])

#### Step 7: Pipeline 4

In [7]:
# Combine all pipelines
pipeline_4 = FeatureUnion(
    [("pipeline_1", pipeline_1), ("pipeline_2", pipeline_2), ("pipeline_3", pipeline_3)]
)
pipeline_4

FeatureUnion(transformer_list=[('pipeline_1',
                                ColumnTransformer(transformers=[('categorical_transformer_1',
                                                                 Pipeline(steps=[('one_hot_transformer',
                                                                                  OneHotEncoder(dtype=<class 'int'>,
                                                                                                sparse_output=False))]),
                                                                 ['NDVI_binary']),
                                                                ('categorical_transformer_2',
                                                                 Pipeline(steps=[('ordinal_transformer',
                                                                                  OrdinalEncoder(categories=[['low_veg',
                                                                                                              'medium_veg',
                                                                                                              'high_veg']]...
                                                                                                   OrdinalEncoder(categories=[['low_veg',
                                                                                                                               'medium_veg',
                                                                                                                               'high_veg']],
                                                                                                                  dtype=<class 'int'>))]),
                                                                                  ['NDVI_categorized']),
                                                                                 ('numerical_transformer',
                                                                                  Pipeline(steps=[('log',
                                                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                                                  ('poly',
                                                                                                   PolynomialFeatures(include_bias=False))]),
                                                                                  ['BLUE',
                                                                                   'GREEN',
                                                                                   'RED',
                                                                                   'NIR',
                                                                                   'SWIR',
                                                                                   'NDVI',
                                                                                   'NDBI',
                                                                                   'REI'])])),
                                                ('min_max_scaler',
                                                 MinMaxScaler()),
                                                ('pca_transformer',
                                                 PCA(n_components=7))]))])

In [8]:
# create a function to select the important features
def select_important_features(X):
    # the important indices are the indices of the top 21 features in the last notebook
    important_indices = [56, 4, 34, 55, 49, 35, 52, 40, 47, 5, 54, 50, 48, 20, 30, 3, 42, 19, 41, 39, 27]
    important_indices = [i - 1 for i in important_indices]
    print("Shape before feature selection:", X.shape)
    return X[:, important_indices]

# Custom transformer to print the shape of the data
def print_shape(X):
    print("Shape after feature selection:", X.shape)
    return X

# Create the final pipeline
classifier = LogisticRegression(solver="lbfgs", max_iter=10000, random_state=1)
select_important_features = FunctionTransformer(select_important_features)

pipeline_5 = Pipeline(steps=[
    ('pipeline_4', pipeline_4),
    ('select_important_features', select_important_features),
    ('print_shape', FunctionTransformer(print_shape, validate=False)),
    ('classifier', classifier)
])
pipeline_5

Pipeline(steps=[('pipeline_4',
                 FeatureUnion(transformer_list=[('pipeline_1',
                                                 ColumnTransformer(transformers=[('categorical_transformer_1',
                                                                                  Pipeline(steps=[('one_hot_transformer',
                                                                                                   OneHotEncoder(dtype=<class 'int'>,
                                                                                                                 sparse_output=False))]),
                                                                                  ['NDVI_binary']),
                                                                                 ('categorical_transformer_2',
                                                                                  Pipeline(steps=[('ordinal_transformer',
                                                                                                   OrdinalEncoder(categories=[['low_...
                                                                                                    'REI'])])),
                                                                 ('min_max_scaler',
                                                                  MinMaxScaler()),
                                                                 ('pca_transformer',
                                                                  PCA(n_components=7))]))])),
                ('select_important_features',
                 FunctionTransformer(func=<function select_important_features at 0x00000210625D8CC0>)),
                ('print_shape',
                 FunctionTransformer(func=<function print_shape at 0x00000210625D8360>)),
                ('classifier',
                 LogisticRegression(max_iter=10000, random_state=1))])

In [9]:
# save the model using pickle
# merge the train and test datasets
X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

# train the model using the whole dataset
pipeline_4.fit(X_all, y_all)
with open('trained_models/selected_features_pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline_4, file)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


END